In [181]:
import sklearn as sk
import pandas as pd
import numpy as np

In [193]:
mida = pd.read_csv("/content/MIDA 5.0.csv") ## Interstate conflict (1816-2014)
nmc =  pd.read_csv("/content/NMC-60-abridged.csv") ## Capabilities of war (1816-2016)
vdem = pd.read_csv("/content/V-Dem-CY-Core-v16.csv") ## Democracy indexes (1789-2025)
orgviol = pd.read_csv("/content/organizedviolencecy_v25_1.csv") # Organize intrastate violence (1989-2024)

In [195]:
# Selecting only important variables and renaming some columns for merging later
mida = mida[["dispnum", "styear", "endyear", "fatality", "fatalpre", "maxdur", "mindur", "hostlev", "recip", "numa", "numb", "ongo2014"]]
nmc = nmc[["stateabb", "ccode", "year", "cinc"]]
vdem = vdem[["country_name", "country_text_id", "year", "v2x_polyarchy", "v2x_libdem", "v2x_partipdem", "v2x_delibdem", "v2x_egaldem"]]
vdem = vdem.rename(columns={"country_text_id" : "stateabb"})
orgviol = orgviol[["country_id_cy", "region_cy", "year_cy", "sb_exist_cy", "sb_dyad_count_cy", "sb_dyad_ids_cy", "sb_dyad_names_cy", "sb_intrastate_exist_cy", "sb_intrastate_dyad_count_cy", "sb_intrastate_dyad_ids_cy", "sb_intrastate_main_govt_inv_incomp_cy", "sb_interstate_exist_cy", "sb_interstate_dyad_count_cy", "sb_interstate_dyad_ids_cy", "sb_interstate_main_govt_inv_incomp_cy", "sb_interstate_main_govt_inv_incomp_cy", "ns_dyad_count_cy", "ns_dyad_ids_cy", "os_exist_cy", "os_dyad_count_cy", "os_dyad_ids_cy", "os_main_govt_inv_cy", "os_nsgroup_inv_cy", "cumulative_total_deaths_parties_in_orgvio_cy", "cumulative_total_deaths_civilians_in_orgvio_cy", "cumulative_total_deaths_unknown_in_orgvio_cy", "cumulative_total_deaths_in_orgvio_best_cy"]]
orgviol = orgviol.rename(columns={"country_id_cy" : "ccode", "year_cy" : "year"})

In [196]:
# orgviol and nmc have same ccode
df1 = pd.merge(nmc, orgviol, on=["year", "ccode"], how="outer")

In [197]:
# Some stateabb are na even though country code exists
ccode_stateabb = df1[["ccode", "stateabb"]].dropna().drop_duplicates()
ccmap = dict(zip(ccode_stateabb["ccode"], ccode_stateabb["stateabb"]))

def get_stateabb(row):
  ccode = row["ccode"]
  if ccode in ccmap:
    return ccmap[ccode]
  else:
    return pd.NA

df1["stateabb"] = df1.apply(get_stateabb, axis=1)

In [198]:
# Combining nmc, vdem and orgviol into one dataframe
df = pd.merge(df1, vdem, on=["year", "stateabb"], how="outer")

stateabbmap = dict(zip(ccode_stateabb["stateabb"], ccode_stateabb["ccode"]))

# Cleaning up some of the missing country codes
def get_ccode(row):
  stateabb = row["stateabb"]
  if stateabb in stateabbmap:
    return stateabbmap[stateabb]
  else:
    return pd.NA

df["ccode"] = df.apply(get_ccode, axis=1)